# CASE 4: Training Reproducibility with TCRα Extension #

Since the number of models generated across the 10 folds is quite large, we provide only the Jupyter notebook for a single fold here, specifically the first fold in the reshuffling process. This notebook reproduces the zero setting of the Meta-learner model. The remaining results can be reproduced by following the same workflow for the other folds.

## classification

### PanPep

##### majority

In [7]:
import sys

SCRIPT_DIR = "../inference/PanPep_Weight_Inference"
TEST_DATA = "../data/independent_dataset(alpha)/tcra_majority.csv"
NEGATIVE_DATA = "../data/independent_dataset(alpha)/Control dataset_tcra.txt" #This is also equivalent to running the rank.
!{sys.executable} {SCRIPT_DIR}/inference_majority.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "../data/fold1-alpha" \
    --support_dir "../data/majority-alpha" \
    --peptide_encoding "{SCRIPT_DIR}/peptide_ab.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_ab.npz" \
    --result_dir "CASE4_majority_rank"

Using 1 GPUs: [0]
Using model configuration: default

Processing peptide: LLAGIGTVPI on device: cuda:0
Positive TCRs: 361, Negative TCRs: 37098
Total batches: 4
Loading support data from: ../data/majority-alpha
Loading support data for peptide: LLAGIGTVPI
Query data size: 36883

Processing batch 1/4 for peptide LLAGIGTVPI - 2026-03-17 15:55:08
Finetuning model...
batch processing time: 13.68s, Progress: 25.0%

Processing batch 2/4 for peptide LLAGIGTVPI - 2026-03-17 15:55:22
Using finetuned model for inference...
batch processing time: 0.53s, Progress: 50.0%

Processing batch 3/4 for peptide LLAGIGTVPI - 2026-03-17 15:55:22
Using finetuned model for inference...
batch processing time: 0.50s, Progress: 75.0%

Processing batch 4/4 for peptide LLAGIGTVPI - 2026-03-17 15:55:23
Using finetuned model for inference...
batch processing time: 0.35s, Progress: 100.0%
Successfully wrote 36883 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE4_majority_rank/LLAGIGTVPI.par

In [8]:
import os
import pandas as pd

PARQUET_DIR = "../inference/PanPep_Weight_Inference/CASE4_majority_rank"
CSV_PATH = "../data/fig5/reshuffling_majority_0.csv"


OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE4_majority_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] ATDALMTGF: 44 rows, 44 scored
[OK] FLCMKALLL: 54 rows, 54 scored
[OK] KTFPPTEPK: 50 rows, 50 scored
[OK] LLAGIGTVPI: 146 rows, 146 scored
[OK] LTDEMIAQY: 48 rows, 48 scored
[OK] QYIKWPWYI: 52 rows, 52 scored
[OK] SPRWYFYYL: 146 rows, 146 scored
[OK] TTDPSFLGRY: 148 rows, 148 scored
[OK] VMATRRNVL: 98 rows, 98 scored
[OK] VMTTVLATL: 42 rows, 42 scored
[OK] YLQPRTFLL: 258 rows, 258 scored

Saved to ./result/CASE4_majority_scored.csv
Total rows: 1086, scored: 1086


In [9]:
!{sys.executable} ../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE4_majority_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [10]:
import pandas as pd

df = pd.read_csv("./CASE4_majority_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE4_majority_scored.csv,0.618303,0.647834,success
1,./result,[FOLDER_SUMMARY],0.618303,0.647834,success (averaged over 1 files)


##### few-shot

In [13]:
import sys

SCRIPT_DIR = "../inference/PanPep_Weight_Inference"
TEST_DATA = "../data/independent_dataset(alpha)/tcra_few.csv"
NEGATIVE_DATA = "../data/independent_dataset(alpha)/Control dataset_tcra.txt" #This is also equivalent to running the rank.
!{sys.executable} {SCRIPT_DIR}/inference_majority.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "../data/fold1-alpha" \
    --support_dir "../data/few-alpha" \
    --peptide_encoding "{SCRIPT_DIR}/peptide_ab.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_ab.npz" \
    --result_dir "CASE4_few_rank"

Using 1 GPUs: [0]
Using model configuration: default

Processing peptide: PSDKHIEQYLKKIQNSLSTEWSP on device: cuda:0
Skipping peptide PSDKHIEQYLKKIQNSLSTEWSP - result file already exists

Processing peptide: GQVELGGGNAVEVCKGS on device: cuda:0
Skipping peptide GQVELGGGNAVEVCKGS - result file already exists

Processing peptide: GPEGAQGPRGEPGTP on device: cuda:0
Skipping peptide GPEGAQGPRGEPGTP - result file already exists

Processing peptide: GQVELGGGNAVEVCK on device: cuda:0
Skipping peptide GQVELGGGNAVEVCK - result file already exists

Processing peptide: HIKEYLNKIQNSLST on device: cuda:0
Skipping peptide HIKEYLNKIQNSLST - result file already exists

Processing peptide: HWFVTQRNFYEPQII on device: cuda:0
Skipping peptide HWFVTQRNFYEPQII - result file already exists

Processing peptide: IHSLLDEGKQSLTKL on device: cuda:0
Skipping peptide IHSLLDEGKQSLTKL - result file already exists

Processing peptide: NCTFEYVSQPFLMDL on device: cuda:0
Skipping peptide NCTFEYVSQPFLMDL - result file alread

In [16]:
import os
import pandas as pd

PARQUET_DIR = "../inference/PanPep_Weight_Inference/CASE4_few_rank"
CSV_PATH = "../data/fig5/reshuffling_few_0.csv"


OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE4_few_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] AAFKRSCLK: 6 rows, 6 scored
[OK] AEVQIDRLI: 10 rows, 10 scored
[OK] AIFYLITPV: 10 rows, 10 scored
[OK] AIIRILQQL: 6 rows, 6 scored
[OK] ALAGIGILTV: 154 rows, 154 scored
[OK] ALDIEIATY: 6 rows, 6 scored
[OK] ALKCKGFHV: 10 rows, 10 scored
[OK] ALSKGVHFV: 32 rows, 32 scored
[OK] ALWEIQQVV: 50 rows, 50 scored
[OK] ALWMRLLPL: 16 rows, 16 scored
[OK] ALWMRLLPLL: 2 rows, 2 scored
[OK] APRGPHGGAASGL: 10 rows, 10 scored
[OK] CLAVHECFV: 12 rows, 12 scored
[OK] CPLSKILL: 16 rows, 16 scored
[OK] DATYQRTRALVR: 186 rows, 186 scored
[OK] DPPALASTNAEVT: 8 rows, 8 scored
[OK] DTDFVNEFY: 8 rows, 8 scored
[OK] EEIEITTHF: 6 rows, 6 scored
[OK] ELAEYLYNI: 6 rows, 6 scored
[OK] ELRSRYWAI: 10 rows, 10 scored
[OK] FLAHIQWMV: 8 rows, 8 scored
[OK] FLALCADSI: 10 rows, 10 scored
[OK] FLHVTYVPA: 6 rows, 6 scored
[OK] FLLNKEMYL: 14 rows, 14 scored
[OK] FLNRFTTTL: 6 rows, 6 scored
[OK] FLPGVYSVI: 6 rows, 6 scored
[OK] FLPRVFSAV: 8 rows, 8 scored
[OK] FLWSVFWLI: 18 rows, 18 scored
[OK] FQQDKHYDL: 8 rows, 8 scor

In [17]:
!{sys.executable} ../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE4_few_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [18]:
import pandas as pd

df = pd.read_csv("./CASE4_few_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE4_few_scored.csv,0.500705,0.416821,success
1,./result,[FOLDER_SUMMARY],0.500705,0.416821,success (averaged over 1 files)


#### zero-shot

In [14]:
import sys

SCRIPT_DIR = "../inference/PanPep_Weight_Inference"
TEST_DATA = "../data/independent_dataset(alpha)/tcra_zero.csv"
NEGATIVE_DATA = "../data/independent_dataset(alpha)/Control dataset_tcra.txt" #This is also equivalent to running the rank.

!{sys.executable} {SCRIPT_DIR}/inference_zero_shot.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "../data/fold1-alpha" \
    --result_dir "CASE4_zero"\
    --peptide_encoding "{SCRIPT_DIR}/peptide_ab.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_ab.npz" \
    --model default

Using 1 GPUs: [0]
Using model configuration: default

Processing peptide: HHLWAYRDLQTRKDIRNAAWHKHGW on device: cuda:0

Processing batch 1/4 for peptide HHLWAYRDLQTRKDIRNAAWHKHGW - 2026-03-17 17:05:41
batch processing time: 0.93s, Progress: 25.0%

Processing batch 2/4 for peptide HHLWAYRDLQTRKDIRNAAWHKHGW - 2026-03-17 17:05:42
batch processing time: 0.57s, Progress: 50.0%

Processing batch 3/4 for peptide HHLWAYRDLQTRKDIRNAAWHKHGW - 2026-03-17 17:05:42
batch processing time: 0.52s, Progress: 75.0%

Processing batch 4/4 for peptide HHLWAYRDLQTRKDIRNAAWHKHGW - 2026-03-17 17:05:43
batch processing time: 0.37s, Progress: 100.0%
Successfully wrote 37459 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE4_zero/HHLWAYRDLQTRKDIRNAAWHKHGW.parquet

Peptide HHLWAYRDLQTRKDIRNAAWHKHGW processing time: 2.48s

Processing peptide: ILYCKRASLTELVSPRLPSHLSEYE on device: cuda:0

Processing batch 1/4 for peptide ILYCKRASLTELVSPRLPSHLSEYE - 2026-03-17 17:05:43
batch processing time: 

In [19]:
import os
import pandas as pd

PARQUET_DIR = "../inference/PanPep_Weight_Inference/CASE4_zero"
CSV_PATH = "../data/fig5/reshuffling_zero_0.csv"

OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE4_zero_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] AALALLLLDRLNQLE: 6 rows, 6 scored
[OK] AALPILFQV: 2 rows, 2 scored
[OK] AAQNVHFWKALNQL: 2 rows, 2 scored
[OK] AAQRIHFFKNLSLL: 2 rows, 2 scored
[OK] AARAVFLAL: 6 rows, 6 scored
[OK] AAVVRFQEAANKQKQ: 4 rows, 4 scored
[OK] ADGLAYFRSSFKGG: 2 rows, 2 scored
[OK] ADTLQSIGATTVASN: 8 rows, 8 scored
[OK] AENPVVHFFKNIATPR: 2 rows, 2 scored
[OK] AFLLFLVLI: 2 rows, 2 scored
[OK] AGSLQPLAL: 2 rows, 2 scored
[OK] AGSLQPLALE: 2 rows, 2 scored
[OK] AHIQWMVMF: 2 rows, 2 scored
[OK] AHPADPGSRPHLIRLFSRDA: 2 rows, 2 scored
[OK] AIILASFSA: 2 rows, 2 scored
[OK] AIKLDDKDP: 2 rows, 2 scored
[OK] AIMTRCLAV: 4 rows, 4 scored
[OK] AISRKMVELVHFLLLKYRAR: 4 rows, 4 scored
[OK] ALALLLLDR: 4 rows, 4 scored
[OK] ALASCMGLI: 2 rows, 2 scored
[OK] ALASCMGLIY: 8 rows, 8 scored
[OK] ALAVLHFYPDKGAKN: 2 rows, 2 scored
[OK] ALDPHSGHFV: 6 rows, 6 scored
[OK] ALEPGPVTA: 2 rows, 2 scored
[OK] ALHGGWTTK: 4 rows, 4 scored
[OK] ALLADKFPV: 2 rows, 2 scored
[OK] ALLETLSLLL: 2 rows, 2 scored
[OK] ALLLQLFTL: 2 rows, 2 scored
[OK]

In [21]:
!{sys.executable} ../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE4_zero_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [22]:
import pandas as pd

df = pd.read_csv("./CASE4_zero_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE4_zero_scored.csv,0.299318,0.266584,success
1,./result,[FOLDER_SUMMARY],0.299318,0.266584,success (averaged over 1 files)


#### Meta-learner

In [15]:
import sys

SCRIPT_DIR = "../inference/PanPep_Weight_Inference"
TEST_DATA = "../data/independent_dataset(alpha)/tcra_zero.csv"
NEGATIVE_DATA = "../data/independent_dataset(alpha)/Control dataset_tcra.txt" #This is also equivalent to running the rank.

!{sys.executable} {SCRIPT_DIR}/inference_meta_learner.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "../data/fold1-alpha" \
    --result_dir "CASE4_meta"\
    --peptide_encoding "{SCRIPT_DIR}/peptide_ab.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_ab.npz" 

Using 1 GPUs: [0]
GPU 0 will process 931 peptides with approximately 1490 TCRs

Processing peptide: ADTLQSIGATTVASN on device: cuda:0
Positive TCRs: 4, Negative TCRs: 37457
Total batches: 4
Query data size: 37461

Processing batch 1/4 for peptide ADTLQSIGATTVASN - 2026-03-17 17:38:09
batch processing time: 0.67s, Progress: 25.0%

Processing batch 2/4 for peptide ADTLQSIGATTVASN - 2026-03-17 17:38:10
batch processing time: 0.40s, Progress: 50.0%

Processing batch 3/4 for peptide ADTLQSIGATTVASN - 2026-03-17 17:38:10
batch processing time: 0.42s, Progress: 75.0%

Processing batch 4/4 for peptide ADTLQSIGATTVASN - 2026-03-17 17:38:11
batch processing time: 0.29s, Progress: 100.0%
Successfully saved results to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE4_meta/ADTLQSIGATTVASN.parquet

Total time for peptide ADTLQSIGATTVASN: 2.16s

Processing peptide: ALASCMGLIY on device: cuda:0
Positive TCRs: 4, Negative TCRs: 37457
Total batches: 4
Query data size: 37461

Processing b

In [23]:
import os
import pandas as pd

PARQUET_DIR = "../inference/PanPep_Weight_Inference/CASE4_meta"
CSV_PATH = "../data/fig5/reshuffling_zero_0.csv"

OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE4_meta_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] AALALLLLDRLNQLE: 6 rows, 6 scored
[OK] AALPILFQV: 2 rows, 2 scored
[OK] AAQNVHFWKALNQL: 2 rows, 2 scored
[OK] AAQRIHFFKNLSLL: 2 rows, 2 scored
[OK] AARAVFLAL: 6 rows, 6 scored
[OK] AAVVRFQEAANKQKQ: 4 rows, 4 scored
[OK] ADGLAYFRSSFKGG: 2 rows, 2 scored
[OK] ADTLQSIGATTVASN: 8 rows, 8 scored
[OK] AENPVVHFFKNIATPR: 2 rows, 2 scored
[OK] AFLLFLVLI: 2 rows, 2 scored
[OK] AGSLQPLAL: 2 rows, 2 scored
[OK] AGSLQPLALE: 2 rows, 2 scored
[OK] AHIQWMVMF: 2 rows, 2 scored
[OK] AHPADPGSRPHLIRLFSRDA: 2 rows, 2 scored
[OK] AIILASFSA: 2 rows, 2 scored
[OK] AIKLDDKDP: 2 rows, 2 scored
[OK] AIMTRCLAV: 4 rows, 4 scored
[OK] AISRKMVELVHFLLLKYRAR: 4 rows, 4 scored
[OK] ALALLLLDR: 4 rows, 4 scored
[OK] ALASCMGLI: 2 rows, 2 scored
[OK] ALASCMGLIY: 8 rows, 8 scored
[OK] ALAVLHFYPDKGAKN: 2 rows, 2 scored
[OK] ALDPHSGHFV: 6 rows, 6 scored
[OK] ALEPGPVTA: 2 rows, 2 scored
[OK] ALHGGWTTK: 4 rows, 4 scored
[OK] ALLADKFPV: 2 rows, 2 scored
[OK] ALLETLSLLL: 2 rows, 2 scored
[OK] ALLLQLFTL: 2 rows, 2 scored
[OK]

In [24]:
!{sys.executable} ../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE4_meta_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [25]:
import pandas as pd

df = pd.read_csv("./CASE4_meta_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE4_meta_scored.csv,0.303142,0.267378,success
1,./result,[FOLDER_SUMMARY],0.303142,0.267378,success (averaged over 1 files)


### Other method

If you would like to reproduce other methods, you can use their web servers. For DLpTCR, you can use http://jianglab.org.cn/DLpTCR/ and select pTCRβ. For ERGO-II, you can use http://tcr2.cs.biu.ac.il/home and choose VDJdb mode.

## rank

For demonstration purposes, since running on the full background pool would be too time-consuming, we constructed a subset pool by randomly sampling 0.1% of the background library and used it for reproduction and result presentation. If you would like to reproduce the full original results, you can simply replace the subset background pool with the complete background library and rerun the same pipeline. In other words, the current setup is only for demonstration efficiency, while the full results can be recovered by using the original background pool.

In [ ]:
!{sys.executable} ../metric_calculation/sort.py \
                    --input_dir ./CASE4_majority_rank \
                    --output_dir ./CASE4_majority_sort

Found 11 files to process:
  CSV files: 0
  Parquet files: 11

Completed processing 11 files


In [ ]:
!{sys.executable} ../metric_calculation/bedroc.py \
                    -i ./CASE4_majority_sort/CASE4_majority_rank \
                    -o ./CASE4_majority_bedroc

2026-03-17 18:23:30,945 - INFO - Input directory: /mnt/d/PanPep_Reusability/jupter/inference/PanPep_Weight_Inference/CASE4_majority_rank
2026-03-17 18:23:30,946 - INFO - Output directory: ./CASE4_majority_bedroc
2026-03-17 18:23:30,946 - INFO - Number of processes: 8
Total: 11 files, To process: 11 files
Processing files:   0%|                                  | 0/11 [00:00<?, ?it/s]
File: ATDALMTGF_sorted.parquet

File: LLAGIGTVPI_sorted.parquet

File: TTDPSFLGRY_sorted.parquet

File: SPRWYFYYL_sorted.parquet

File: KTFPPTEPK_sorted.parquet

File: LTDEMIAQY_sorted.parquet

File: QYIKWPWYI_sorted.parquet

File: FLCMKALLL_sorted.parquet
Processing files:   9%|██▎                       | 1/11 [00:00<00:01,  7.89it/s]
File: VMATRRNVL_sorted.parquet

File: VMTTVLATL_sorted.parquet

File: YLQPRTFLL_sorted.parquet
Processing files: 100%|█████████████████████████| 11/11 [00:00<00:00, 64.35it/s]
2026-03-17 18:23:31,200 - INFO - Successfully processed 11 files.
2026-03-17 18:23:31,200 - INFO - 

In [ ]:
import os
import glob
import pandas as pd

detailed_dir = "./CASE4_majority_bedroc"

# Print all CSV files in the detailed results directory
csv_files = sorted(glob.glob(os.path.join(detailed_dir, "*.csv")))

print(f"Found {len(csv_files)} CSV file(s) in: {detailed_dir}\n")

for file_path in csv_files:
    print("=" * 100)
    print(f"File: {file_path}")
    print("=" * 100)
    df = pd.read_csv(file_path)
    print(df)
    print("\n")

Found 12 CSV file(s) in: /mnt/d/PanPep_Reusability/jupter/CASE4_majority_bedroc

File: /mnt/d/PanPep_Reusability/jupter/CASE4_majority_bedroc/summary_by_directory.csv
                                           Directory  BEDROC_Mean  BEDROC_Std  \
0  /mnt/d/PanPep_Reusability/jupter/inference/Pan...     0.119206    0.121052   

   File_Count  
0          11  


File: /mnt/d/PanPep_Reusability/jupter/CASE4_majority_bedroc/temp_ATDALMTGF_sorted.csv
                                           Directory  \
0  /mnt/d/PanPep_Reusability/jupter/inference/Pan...   

                   Filename  BEDROC_Score  Alpha  
0  ATDALMTGF_sorted.parquet      0.143054     20  


File: /mnt/d/PanPep_Reusability/jupter/CASE4_majority_bedroc/temp_FLCMKALLL_sorted.csv
                                           Directory  \
0  /mnt/d/PanPep_Reusability/jupter/inference/Pan...   

                   Filename  BEDROC_Score  Alpha  
0  FLCMKALLL_sorted.parquet      0.026134     20  


File: /mnt/d/PanPep_Reusabil

In [ ]:
!{sys.executable} "../metric_calculation/success_rate&hit_rate.py" \
    --root_dir "./CASE4_majority_sort/CASE4_majority_rank" \
    --output "CASE4_majority_sort" \
    --output_dir "./CASE4_majority_success"

2026-03-17 18:27:31,066 - INFO - Multiprocessing start method set to 'spawn'.
2026-03-17 18:27:31,169 - INFO - Configuration: ProcessingConfig(root_dir='/mnt/d/PanPep_Reusability/jupter/inference/PanPep_Weight_Inference/CASE4_majority_rank', top_k_ratio=1, batch_size=150, num_gpus=1, output_file='CASE4_majority_sort', output_dir='./CASE4_majority_success', gpu_id=None)
2026-03-17 18:27:31,169 - INFO - Detected 1 GPU(s)
Processing directories: 100%|█████████████████████| 1/1 [00:02<00:00,  2.26s/it]
2026-03-17 18:27:33,466 - INFO - Combining 1 DataFrames...
2026-03-17 18:27:33,663 - INFO - Final results saved to CASE4_majority_success/CASE4_majority_sort.csv and CASE4_majority_success/CASE4_majority_sort.parquet


In [ ]:
!{sys.executable} "../metric_calculation/get_success_AUC.py" \
    --input "./CASE4_majority_success/CASE4_majority_sort.parquet" \
    --output "CASE4_majority_success_AUC" 

Found 1 directories.
1. '/mnt/d/PanPep_Reusability/jupter/inference/PanPep_Weight_Inference/CASE4_majority_rank'
['/mnt/d/PanPep_Reusability/jupter/inference/PanPep_Weight_Inference/CASE4_majority_rank'] n=37299  Top_K range=(1, 37299)  y range=(0, 1)
/mnt/d/PanPep_Reusability/jupter/../metric_calculation/get_success_AUC.py:190: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  area = float(np.trapz(ys, xs_norm))
  AUC@0.0100%: area=0
  AUC@0.1000%: area=3.91939e-06
  AUC@1.0000%: area=0.000224162
  AUC@5.0000%: area=0.00386248
  AUC@10.0000%: area=0.0120037
  AUC@20.0000%: area=0.0409374
  AUC@100.0000%: area=0.603073

Saved results to: CASE4_majority_success_AUC
